# REDD Dataset Exploratory Analysis
This notebook explores the REDD pickled datasets (train/val/test). It inspects shapes, summary statistics, and visualizes sample distributions and time-series slices when available.

In [5]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

file_path = "data/redd/test_small.pkl"

try:
    with open(file_path, "rb") as f:
        data = pickle.load(f)

    print(f"Successfully loaded {file_path}")
    print(f"Type of data: {type(data)}")

    # Attempt to print a portion of the data, depending on its type
    if isinstance(data, dict):
        print(f"Keys: {list(data.keys())}")
        for key, value in data.items():
            print(f"--- Key: {key} ---")
            print(f"Type of value: {type(value)}")
            if hasattr(value, '__len__'):
                print(f"Length of value: {len(value)}")
            if isinstance(value, dict):
                print(f"Sub-keys: {list(value.keys())}")
            elif isinstance(value, list) and len(value) > 0:
                print(f"First element type: {type(value[0])}")
                print(f"First element: {value[0]}")
            elif isinstance(value, pd.DataFrame):
                print(f"DataFrame head:\n{value.head()}\n")
            elif hasattr(value, 'shape'):
                print(f"Shape of value: {value.shape}")
                sample = value if value.size < 10 else value.flatten()[:5]
                print(f"First few elements: {sample}")
            else:
                print(f"Value: {value}")
            print("\n")
    elif isinstance(data, list):
        print(f"Length of list: {len(data)}")
        if len(data) > 0:
            print(f"Type of first element: {type(data[0])}")
            print(f"First element: {data[0]}")
            if hasattr(data[0], 'shape'):
                print(f"Shape of first element: {data[0].shape}")
            if isinstance(data[0], pd.DataFrame):
                print(f"DataFrame head:\n{data[0].head()}\n")
    elif isinstance(data, pd.DataFrame):
        print(f"DataFrame head:\n{data.head()}\n")
    elif hasattr(data, 'shape'):
        print(f"Shape of data: {data.shape}")
        sample = data if data.size < 10 else data.flatten()[:5]
        print(f"First few elements: {sample}")
    else:
        print(f"Data: {data}")

except Exception as e:
    print(f"Error loading or inspecting the pkl file: {e}")

Successfully loaded data/redd/test_small.pkl
Type of data: <class 'list'>
Length of list: 6
Type of first element: <class 'pandas.core.frame.DataFrame'>
First element:                             main  dish washer  fridge  microwave  washer dryer
2011-04-18 09:22:12-04:00  261.0          0.0     6.0        5.0           0.0
2011-04-18 09:22:15-04:00  261.0          0.0     6.0        5.0           0.0
2011-04-18 09:22:18-04:00  262.0          0.0     6.0        5.0           0.0
2011-04-18 09:22:21-04:00  262.0          1.0     6.0        5.0           0.0
2011-04-18 09:22:24-04:00  261.0          0.0     6.0        5.0           0.0
...                          ...          ...     ...        ...           ...
2011-04-19 18:44:54-04:00  307.0          0.0     6.0        5.0           0.0
2011-04-19 18:44:57-04:00  308.0          0.0     6.0        4.0           0.0
2011-04-19 18:45:00-04:00  304.0          0.0     7.0        5.0           0.0
2011-04-19 18:45:03-04:00  304.0          

In [6]:
from pathlib import Path

split_files = {
    "train": Path("data/redd/train_small.pkl"),
    "val": Path("data/redd/val_small.pkl"),
    "test": Path("data/redd/test_small.pkl"),
}

splits = {}
for name, path in split_files.items():
    if not path.exists():
        print(f"Missing split file: {path}")
        continue
    with open(path, "rb") as f:
        splits[name] = pickle.load(f)
    print(f"Loaded {name}: {path} -> {type(splits[name])}")

print(f"Available splits: {list(splits.keys())}")

Loaded train: data\redd\train_small.pkl -> <class 'list'>
Loaded val: data\redd\val_small.pkl -> <class 'list'>
Loaded test: data\redd\test_small.pkl -> <class 'list'>
Available splits: ['train', 'val', 'test']


In [7]:
def _array_summary(arr):
    arr = np.asarray(arr)
    arr = arr.reshape(-1)
    return {
        "shape": tuple(arr.shape),
        "mean": float(np.mean(arr)),
        "std": float(np.std(arr)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "nonzero_frac": float(np.count_nonzero(arr)) / len(arr) if len(arr) else 0.0,
    }

def _find_first_key(mapping, candidates):
    for key in candidates:
        if key in mapping:
            return key
    return None

def summarize_split(split_name, payload):
    print(f"\n=== {split_name.upper()} SUMMARY ===")
    if isinstance(payload, dict):
        print(f"Keys: {list(payload.keys())}")
        mains_key = _find_first_key(payload, ["mains", "aggregate", "X", "x", "inputs", "input", "power"])
        target_key = _find_first_key(payload, ["targets", "y", "Y", "appliance", "appliances"])

        if mains_key is not None:
            mains = payload[mains_key]
            if hasattr(mains, "shape") or isinstance(mains, list):
                try:
                    print(f"Mains key '{mains_key}' summary: {_array_summary(mains)}")
                except Exception as exc:
                    print(f"Could not summarize mains ({mains_key}): {exc}")

        if target_key is not None:
            target = payload[target_key]
            if isinstance(target, dict):
                for appliance, values in target.items():
                    try:
                        print(f"Target '{appliance}' summary: {_array_summary(values)}")
                    except Exception as exc:
                        print(f"Could not summarize target {appliance}: {exc}")
            elif hasattr(target, "shape") or isinstance(target, list):
                try:
                    print(f"Target key '{target_key}' summary: {_array_summary(target)}")
                except Exception as exc:
                    print(f"Could not summarize target ({target_key}): {exc}")
    else:
        if hasattr(payload, "shape") or isinstance(payload, list):
            try:
                print(f"Array summary: {_array_summary(payload)}")
            except Exception as exc:
                print(f"Could not summarize payload: {exc}")

for name, payload in splits.items():
    summarize_split(name, payload)


=== TRAIN SUMMARY ===
Array summary: {'shape': (144000,), 'mean': 150.57247924804688, 'std': 576.6075439453125, 'min': 0.0, 'max': 7604.5, 'nonzero_frac': 0.719875}

=== VAL SUMMARY ===
Array summary: {'shape': (144000,), 'mean': 210.13609313964844, 'std': 786.95263671875, 'min': 0.0, 'max': 7425.5, 'nonzero_frac': 0.7013541666666666}

=== TEST SUMMARY ===
Could not summarize payload: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (6,) + inhomogeneous part.


In [9]:
def plot_distribution(arr, title, max_points=20000):
    arr = np.asarray(arr).reshape(-1)
    if len(arr) == 0:
        print(f"No data to plot for {title}")
        return
    if len(arr) > max_points:
        rng = np.random.default_rng(42)
        arr = rng.choice(arr, size=max_points, replace=False)
    plt.figure(figsize=(7, 4))
    sns.histplot(arr, bins=50, kde=True)
    plt.title(title)
    plt.xlabel("Power (W)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

def plot_timeseries(arr, title, points=1000):
    arr = np.asarray(arr).reshape(-1)
    if len(arr) == 0:
        print(f"No data to plot for {title}")
        return
    plt.figure(figsize=(9, 3))
    plt.plot(arr[:points])
    plt.title(title)
    plt.xlabel("Time step")
    plt.ylabel("Power (W)")
    plt.tight_layout()
    plt.show()

for name, payload in splits.items():
    if not isinstance(payload, dict):
        continue
    mains_key = _find_first_key(payload, ["mains", "aggregate", "X", "x", "inputs", "input", "power"])
    target_key = _find_first_key(payload, ["targets", "y", "Y", "appliance", "appliances"])

    if mains_key is not None:
        mains = payload[mains_key]
        try:
            plot_distribution(mains, f"{name.upper()} mains distribution ({mains_key})")
            plot_timeseries(mains, f"{name.upper()} mains sample ({mains_key})")
        except Exception as exc:
            print(f"Unable to plot mains for {name}: {exc}")

    if target_key is not None:
        target = payload[target_key]
        if isinstance(target, dict):
            for appliance, values in target.items():
                try:
                    plot_distribution(values, f"{name.upper()} {appliance} distribution")
                    plot_timeseries(values, f"{name.upper()} {appliance} sample")
                except Exception as exc:
                    print(f"Unable to plot {appliance} for {name}: {exc}")
        else:
            try:
                plot_distribution(target, f"{name.upper()} target distribution ({target_key})")
                plot_timeseries(target, f"{name.upper()} target sample ({target_key})")
            except Exception as exc:
                print(f"Unable to plot targets for {name}: {exc}")